# inat2symbiota

This notebook downloads iNaturalist observation data and converts them into a CSV file in the format expected for ingestion by Synbiota.
Input is a list of integers containing iNaturalist observation numbers.

Example:

inet_obs_to_fetch = [386506396, 385901816, 385901810]

## Prompt:

I have an iNaturalist observation of an insects with an images and metadata. A specimen associated with each observation is stored in an insect collection which is catalogued using Symbiota. Write a python script to harvest iNaturalist records and convert them into the format expected for ingestion by Symbiota,

## Response

Below is a Python script that harvests insect observations from the iNaturalist API (using requests), maps their metadata and image URLs into standard Darwin Core / Symbiota import fields, and outputs a UTF-8 CSV ready for Symbiota's batch import process.

### Standard Mapping Breakdown

Symbiota Field | iNaturalist Attribute | Notes
-------------- | --------------------- | -----
| occurrenceID | `https://www.inaturalist.org/observations<id>` | Unique permalink / persistent identifier.
catalogNumber | Observation ID / Field Value | Internal collection barcode/catalog number (if stored in an observation field) or default iNat ID.
basisOfRecord | "PreservedSpecimen" | Identifies the physical specimen in your collection.
recordedBy | Observer username / Full nameCollector name.
eventDate | observed_on / time_observed_at | Date in YYYY-MM-DD ISO format.
scientificName | Taxon name | Complete scientific name.
decimalLatitude / Longitude | location coordinates | Georeference coordinates.
coordinateUncertaintyInMeters | positional_accuracy | Coordinate precision in meters.
associatedMediaPhoto URLs (semicolon-separated) | Direct links to original/large size images. 

### Ingestion into Symbiota

1. Open your Symbiota portal and navigate to Collection Management $\rightarrow$ Import/Update Specimen Records.
2. Choose Full Text File Upload.
3. Upload symbiota_insect_import.csv.
4. Map the columns during the step-by-step prompt (Symbiota auto-detects standard Darwin Core terms like catalogNumber, recordedBy, eventDate, and associatedMedia).
5. Complete the temporary transfer verification, then transfer the records to the central specimen database.

## Notes

1. I had a problem opening the CSV file with Libre Office Calc. The URLs for images were being placed in separate columns. For some reason `Semicolon` was selected as a `Separator Options`. Deselecting this option fixed the problem.

In [41]:
import csv
import logging
from  typing import Optional, Any
import requests

In [42]:
# Configure basic logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [43]:
inet_obs_to_fetch = [386506396, 385901816, 385901810]

In [44]:
def fetch_inaturalist_observations(
    user_id: Optional[str] = None,
    taxon_id: Optional[int] = 47158,  # Default: Insecta
    project_id: Optional[str] = None,
    observation_ids: Optional[list[int]] = None,
    max_results: int = 100,
) -> list[dict[str, Any]]:
    """Harvests observation records and associated metadata from the iNaturalist v1 API.

    Args:
        user_id: iNaturalist username or user numerical ID to filter observations.
        taxon_id: iNaturalist taxon ID (defaults to 47158 for Class Insecta).
        project_id: iNaturalist project slug or ID to filter observations.
        observation_ids: List of specific iNaturalist observation IDs to retrieve.
        max_results: Maximum total number of observations to fetch.

    Returns:
        A list of raw observation dictionaries returned by the iNaturalist API.

    Raises:
        requests.HTTPError: If the API request fails due to an HTTP error status code.
    """
    endpoint = "https://api.inaturalist.org/v1/observations"
    observations: List[Dict[str, Any]] = []
    page = 1
    per_page = min(max_results, 200)

    while len(observations) < max_results:
        params: Dict[str, Any] = {
            "per_page": per_page,
            "page": page,
            "order": "desc",
            "order_by": "created_at",
        }

        if user_id:
            params["user_id"] = user_id
        if taxon_id:
            params["taxon_id"] = taxon_id
        if project_id:
            params["project_id"] = project_id
        if observation_ids:
            params["id"] = ",".join(map(str, observation_ids))

        logger.info("Fetching iNaturalist records (Page %d)...", page)
        response = requests.get(endpoint, params=params, timeout=15)
        response.raise_for_status()

        data = response.json()
        results = data.get("results", [])

        if not results:
            break

        observations.extend(results)

        if len(results) < per_page or len(observations) >= max_results:
            break

        page += 1

    logger.info("Retrieved %d observations from iNaturalist.", len(observations))
    return observations[:max_results]



In [45]:

def transform_to_symbiota_record(obs: dict[str, Any]) -> dict[str, Any]:
    """Transforms a single iNaturalist observation API payload into a Symbiota-compatible dict.

    Args:
        obs: A raw observation dictionary from the iNaturalist API.

    Returns:
        A flat dictionary keyed by standard Symbiota / Darwin Core import field names.
    """
    obs_id = obs.get("id")
    taxon_data = obs.get("taxon") or {}
    user_data = obs.get("user") or {}

    # Extract location coordinates
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    if obs.get("location"):
        coords = obs["location"].split(",")
        if len(coords) == 2:
            try:
                latitude = float(coords[0])
                longitude = float(coords[1])
            except ValueError:
                pass

    # Extract collector name (prefer full name if present, fallback to login handle)
    collector = user_data.get("name") or user_data.get("login") or ""

    # Aggregate media links (replace 'square' image thumb with 'large' resolution)
    photo_urls: List[str] = []
    for photo_obj in obs.get("photos", []):
        url = photo_obj.get("url", "")
        if url:
            large_url = url.replace("square.", "large.").replace("medium.", "large.")
            photo_urls.append(large_url)

    # Convert photos list into a semicolon-delimited string
    associated_media = ";".join(photo_urls)

    # Extract taxonomy hierarchy where available
    scientific_name = taxon_data.get("name", "")
    rank = taxon_data.get("rank", "")

    return {
        "dbpk": f"iNat-{obs_id}",
        "catalogNumber": f"iNat-{obs_id}",
        "occurrenceID": f"https://www.inaturalist.org/observations/{obs_id}",
        "basisOfRecord": "PreservedSpecimen",
        "scientificName": scientific_name,
        "taxonRank": rank,
        "kingdom": "Animalia",
        "phylum": "Arthropoda",
        "class": "Insecta",
        "recordedBy": collector,
        "eventDate": obs.get("observed_on", ""),
        "verbatimEventDate": obs.get("time_observed_at", ""),
        "country": obs.get("place_guess", ""),
        "decimalLatitude": latitude if latitude is not None else "",
        "decimalLongitude": longitude if longitude is not None else "",
        "coordinateUncertaintyInMeters": obs.get("positional_accuracy", ""),
        "occurrenceRemarks": obs.get("description", ""),
        "associatedMedia": associated_media,
        "dataGeneralizations": f"Harvested from iNaturalist observation {obs_id}",
    }


In [46]:
def export_symbiota_csv(records: list[dict[str, Any]], output_filepath: str) -> None:
    """Writes a list of transformed Symbiota records to a CSV file.

    Args:
        records: List of dictionaries representing mapped Symbiota records.
        output_filepath: Target filesystem path for the output CSV file.

    Returns:
        None
    """
    if not records:
        logger.warning("No records to export. Output CSV was not written.")
        return

    fieldnames = list(records[0].keys())

    with open(output_filepath, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    logger.info("Successfully exported %d records to '%s'.", len(records), output_filepath)


In [47]:


def main() -> None:
    """Main execution block to demonstrate harvesting and converting iNaturalist data."""
    # Example usage configuration:
    target_username: Optional[str] = None  # e.g., "my_username"
    insect_taxon_id: int = 47158  # Insecta
    output_filename: str = "symbiota_insect_import.csv"

    logger.info("Starting iNaturalist harvest for Symbiota conversion...")

    raw_observations = fetch_inaturalist_observations(observation_ids=inet_obs_to_fetch)  # Example observation IDs

    symbiota_records = [transform_to_symbiota_record(obs) for obs in raw_observations]

    export_symbiota_csv(symbiota_records, output_filename)


if __name__ == "__main__":
    main()



2026-08-04 08:36:00,777 - INFO - Starting iNaturalist harvest for Symbiota conversion...
2026-08-04 08:36:00,778 - INFO - Fetching iNaturalist records (Page 1)...
2026-08-04 08:36:02,162 - INFO - Retrieved 3 observations from iNaturalist.
2026-08-04 08:36:02,164 - INFO - Successfully exported 3 records to 'symbiota_insect_import.csv'.
